In [22]:
import jax
from jax import numpy as jnp
import flax.linen as nn

# jax ravel
from jax.flatten_util import ravel_pytree

def sequential_initializer(start=1):
    def init(key, shape, dtype=jnp.float32):
        size = 1
        for s in shape:
            size *= s
        return (jnp.arange(size, dtype=dtype) + start).reshape(shape)
    return init

class SimpleMLP(nn.Module):
    features: list

    @nn.compact
    def __call__(self, x):
        seq_init = sequential_initializer(1)

        for feat in self.features[:-1]:
            x = nn.Dense(
                feat,
                kernel_init=seq_init,
                bias_init=seq_init
            )(x)
            x = nn.relu(x)

        x = nn.Dense(
            self.features[-1],
            kernel_init=seq_init,
            bias_init=seq_init
        )(x)

        return x
    
def pretty_print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key) + ':', end=' ')
        if isinstance(value, dict):
            print()
            pretty_print_dict(value, indent + 1)
        else:
            print(value)

N_input = 2
N_batch = 16
model = SimpleMLP(features=[4,2])
key1, key2 = jax.random.split(jax.random.PRNGKey(0), 2)
x = jnp.linspace(-1, 1, N_batch*N_input).reshape((N_batch, N_input))  # Example input with batch size 8 and feature size 4
params = model.init(key2, x)
y = model.apply(params, x)

array_params = ravel_pytree(params)

print("input shape:", x.shape)
print("Output shape:", y.shape)
print("Number of parameters:", array_params[0].shape)
print("Computed number of parameters:", sum(jnp.prod(jnp.array(p.shape)) for p in jax.tree_util.tree_leaves(params)))
print("Parameters (raveled):", array_params[0])
print("Parameters (original):")
pretty_print_dict(params)

input shape: (16, 2)
Output shape: (16, 2)
Number of parameters: (22,)
Computed number of parameters: 22
Parameters (raveled): [1. 2. 3. 4. 1. 2. 3. 4. 5. 6. 7. 8. 1. 2. 1. 2. 3. 4. 5. 6. 7. 8.]
Parameters (original):
params: 
  Dense_0: 
    kernel: [[1. 2. 3. 4.]
 [5. 6. 7. 8.]]
    bias: [1. 2. 3. 4.]
  Dense_1: 
    kernel: [[1. 2.]
 [3. 4.]
 [5. 6.]
 [7. 8.]]
    bias: [1. 2.]


In [ ]:
def test_function_with_subfunctions(common_param):
    def subfunction_a(x):
        return x + common_param

    def subfunction_b(x):
        return x * common_param

    return subfunction_a, subfunction_b

